# Semana 47 · ¿Cómo sabemos que hay planetas en otras estrellas?

Este cuaderno acompaña a la sesión de proyecto de la semana 47: buscar un planeta en tránsito escondido en el ruido de una curva de luz, medirlo y pesarlo con la velocidad radial. **Los datos principales son simulados**, con propiedades parecidas a los de la misión TESS; al final hay una opción, no probada, para descargar datos reales.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar; no puedes estropear nada.

Tiempo estimado: 30-45 minutos.

## 1. Una curva de luz simulada

Una estrella como el Sol observada durante 27 días (lo que TESS observa cada zona del cielo), con una medida cada 2 minutos y un ruido del 0,1 % en cada medida. Escondemos un planeta: el cuaderno elige su periodo, su radio y el momento del primer tránsito al azar, y no los mira hasta el final.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(47)
dias, cadencia = 27.0, 2 / 60 / 24            # 2 minutos en días
t = np.arange(0, dias, cadencia)
ruido = 0.001

# el planeta escondido (no mires estas líneas hasta el final)
P_real = rng.uniform(2.0, 6.0)
t0_real = rng.uniform(0, P_real)
Rp_Rs_real = rng.uniform(0.04, 0.07)
duracion_real = 0.12 * (P_real / 3) ** (1 / 3)   # días, aproximada para una estrella como el Sol

def modelo_caja(t, P, t0, profundidad, duracion):
    fase = ((t - t0 + 0.5 * P) % P) - 0.5 * P   # tiempo respecto al centro del tránsito más cercano
    return 1 - profundidad * (np.abs(fase) < duracion / 2)

flujo = modelo_caja(t, P_real, t0_real, Rp_Rs_real**2, duracion_real) + rng.normal(0, ruido, t.size)
flujo *= 1 + 0.0005 * np.sin(2 * np.pi * t / 11)    # una variación lenta de la estrella (manchas que giran)

plt.figure(figsize=(12, 3.5))
plt.plot(t, flujo, ".", ms=1)
plt.xlabel("tiempo (días)"); plt.ylabel("flujo relativo"); plt.title("Curva de luz simulada (27 días)"); plt.show()
print(f"{t.size} medidas")

A simple vista quizá se intuyan algunas caídas, o quizá no. Primero quitamos la variación lenta de la estrella: dividimos por una **mediana móvil** de un día, que sigue los cambios lentos pero apenas nota los tránsitos, que duran unas horas. (La mediana es mejor que la media para esto, porque los pocos puntos del tránsito casi no la mueven: semana 39.)

In [ ]:
import pandas as pd
ventana = int(1.0 / cadencia)                        # un día de medidas
suave = pd.Series(flujo).rolling(ventana, center=True).median().to_numpy()
# en los bordes la ventana está incompleta y da NaN: los recortamos
dentro = ~np.isnan(suave)
t_l, f_l = t[dentro], (flujo / suave)[dentro]

plt.figure(figsize=(12, 3.5))
plt.plot(t_l, f_l, ".", ms=1)
plt.xlabel("tiempo (días)"); plt.ylabel("flujo normalizado"); plt.title("Tras quitar la variación lenta"); plt.show()

## 2. Buscar el periodo con un ajuste de caja

Para cada periodo de prueba, plegamos los datos y buscamos la caja (posición y duración) que más baja el brillo. Medimos lo bien que encaja con el cociente señal-ruido de la caja: la caída media dentro de la caja dividida por su incertidumbre, $\sigma/\sqrt{N}$. Para ir deprisa, agrupamos antes la curva plegada en intervalos de fase.

In [ ]:
def mejor_caja(t, f, P, duraciones=(0.06, 0.08, 0.10, 0.12, 0.14, 0.16, 0.2), n_bins=400):
    fase = (t % P) / P
    sumas = np.bincount((fase * n_bins).astype(int), weights=1 - f, minlength=n_bins)
    cuentas = np.bincount((fase * n_bins).astype(int), minlength=n_bins)
    mejor = (0, 0, 0, 0)
    for dur in duraciones:
        ancho = max(1, int(round(dur / P * n_bins)))
        s = np.convolve(np.r_[sumas, sumas[:ancho]], np.ones(ancho), "valid")[:n_bins]
        n = np.convolve(np.r_[cuentas, cuentas[:ancho]], np.ones(ancho), "valid")[:n_bins]
        prof = s / np.maximum(n, 1)
        snr = prof * np.sqrt(n) / ruido
        i = np.argmax(snr)
        if snr[i] > mejor[0]:
            mejor = (snr[i], prof[i], (i + ancho / 2) / n_bins * P, dur)
    return mejor

periodos = np.linspace(1.0, 10.0, 4000)
resultados = [mejor_caja(t_l, f_l, P) for P in periodos]
snr = np.array([r[0] for r in resultados])
P_mejor = periodos[np.argmax(snr)]

plt.figure(figsize=(10, 4))
plt.plot(periodos, snr)
plt.axvline(P_mejor, color="r", ls="--", lw=0.8)
for k in [0.5, 2]:
    plt.axvline(P_mejor * k, color="gray", ls=":", lw=0.8)
plt.xlabel("periodo de prueba (días)"); plt.ylabel("señal / ruido de la mejor caja"); plt.title("Periodograma de ajuste de caja")
plt.grid(alpha=0.3); plt.show()
print(f"periodo con la mejor señal: {P_mejor:.3f} días")

El pico más alto marca el periodo. Las líneas grises punteadas señalan la mitad y el doble: ahí suele haber picos secundarios.

## 3. Plegar y medir

In [ ]:
snr_m, prof_m, centro_fase, dur_m = mejor_caja(t_l, f_l, P_mejor)
fase_t = ((t_l - centro_fase + 0.5 * P_mejor) % P_mejor) - 0.5 * P_mejor     # horas respecto al centro
horas = fase_t * 24

bins = np.linspace(-8, 8, 97)
idx = np.digitize(horas, bins)
medias = np.array([f_l[idx == i].mean() if np.any(idx == i) else np.nan for i in range(1, len(bins))])

plt.figure(figsize=(10, 4.5))
plt.plot(horas, f_l, ".", ms=1, alpha=0.3, label="medidas plegadas")
plt.plot(0.5 * (bins[1:] + bins[:-1]), medias, "ro-", ms=3, label="media cada 10 minutos")
plt.xlim(-8, 8); plt.xlabel("horas desde el centro del tránsito"); plt.ylabel("flujo"); plt.legend(); plt.grid(alpha=0.3)
plt.show()
# una medida más robusta de la profundidad: mediana fuera del tránsito menos mediana en su parte central
fuera = np.abs(horas) > 1.5 * dur_m * 24
centro = np.abs(horas) < 0.3 * dur_m * 24
prof_m = np.median(f_l[fuera]) - np.median(f_l[centro])
print(f"profundidad ≈ {prof_m * 100:.3f} %   duración ≈ {dur_m * 24:.1f} h   señal/ruido ≈ {snr_m:.0f}")

## 4. El planeta

Con la profundidad, $R_p/R_\star = \sqrt{\delta}$; con una estrella como el Sol, el radio del planeta. Con el periodo y la tercera ley de Kepler, el tamaño de la órbita. Y ahora sí, comparamos con lo que el cuaderno había escondido.

In [ ]:
R_sol_km, R_tierra_km = 696_000, 6_371
Rp = np.sqrt(prof_m) * R_sol_km
a_ua = (P_mejor / 365.25) ** (2 / 3)                 # Kepler, con la masa del Sol
print(f"radio del planeta ≈ {Rp:,.0f} km = {Rp / R_tierra_km:.1f} radios terrestres".replace(",", " "))
print(f"órbita: a ≈ {a_ua:.3f} ua")
print()
print(f"lo escondido: P = {P_real:.3f} días, R_p = {Rp_Rs_real * R_sol_km / R_tierra_km:.1f} radios terrestres, duración {duracion_real * 24:.1f} h")

Para la profundidad no usamos la de la caja, que mezcla algo de fuera del tránsito si no encaja perfectamente, sino la diferencia entre las medianas de fuera y de la parte central del tránsito. Los ajustes profesionales usan un modelo de tránsito con su forma real (con bordes inclinados y el oscurecimiento del borde de la estrella).

## 5. Pesar el planeta con la velocidad radial

Simulamos 25 medidas de la velocidad radial de la estrella, repartidas al azar durante dos meses, con un error de 3 m/s, para un planeta de 0,5 masas de Júpiter con el periodo encontrado. Ajustamos una sinusoide con el periodo fijo: $v(t) = K\sin(2\pi(t - t_1)/P)$, que se puede escribir como $A\sin(2\pi t/P) + B\cos(2\pi t/P)$, un modelo **lineal** en $A$ y $B$ que se ajusta con mínimos cuadrados como una recta.

In [ ]:
m_real = 0.5                                                  # masas de Júpiter
K_real = 28.4 * m_real * (P_mejor / 365.25) ** (-1 / 3)
t_rv = np.sort(rng.uniform(0, 60, 25))
v_rv = K_real * np.sin(2 * np.pi * (t_rv - t0_real) / P_mejor) + rng.normal(0, 3, t_rv.size)

X = np.column_stack([np.sin(2 * np.pi * t_rv / P_mejor), np.cos(2 * np.pi * t_rv / P_mejor)])
(A, B), *_ = np.linalg.lstsq(X, v_rv, rcond=None)
K = np.hypot(A, B)
m_sin_i = K / (28.4 * (P_mejor / 365.25) ** (-1 / 3))

fase_rv = (t_rv % P_mejor) / P_mejor
fase_fina = np.linspace(0, 1, 300)
plt.figure(figsize=(9, 4.5))
plt.errorbar(fase_rv, v_rv, yerr=3, fmt="o", ms=4, label="medidas simuladas")
plt.plot(fase_fina, A * np.sin(2 * np.pi * fase_fina) + B * np.cos(2 * np.pi * fase_fina), "r", label=f"ajuste: K = {K:.1f} m/s")
plt.xlabel("fase orbital"); plt.ylabel("velocidad radial (m/s)"); plt.legend(); plt.grid(alpha=0.3); plt.show()
print(f"masa mínima: m sin i ≈ {m_sin_i:.2f} masas de Júpiter (simulada: {m_real})")

Como el planeta transita, sabemos que su órbita está casi de canto, $\sin i \approx 1$, y la masa mínima es casi la masa real. Con la masa y el radio, la densidad.

## 6. Opcional: una curva de luz real de TESS (sin probar)

La biblioteca `lightkurve` descarga curvas de luz públicas de Kepler y TESS desde el archivo MAST de la NASA. **Esta celda no se ha podido probar** al escribir el cuaderno. Cambia `USAR_TESS` a `True` en Colab. Como ejemplo usa la estrella Pi Mensae, alrededor de la cual TESS descubrió en 2018 su primer planeta, con un periodo de unos 6,3 días; si la búsqueda no devuelve datos, prueba con otra estrella con planetas conocidos.

In [ ]:
USAR_TESS = False

if USAR_TESS:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightkurve"])
    import lightkurve as lk
    busqueda = lk.search_lightcurve("Pi Mensae", mission="TESS", author="SPOC", exptime=120)
    print(busqueda)
    curva = busqueda[0].download().remove_nans().normalize().flatten(window_length=901)
    curva.plot()
    pg = curva.to_periodogram(method="bls", period=np.linspace(1, 15, 5000))
    print(f"periodo con más potencia: {pg.period_at_max_power}")
    curva.fold(period=pg.period_at_max_power, epoch_time=pg.transit_time_at_max_power).scatter()
else:
    print("Descarga de TESS desactivada. Cambia USAR_TESS a True para probarla.")

## 7. Experimenta

1. Haz el planeta más pequeño (por ejemplo, $R_p/R_\star = 0{,}02$, como una supertierra) y vuelve a buscarlo. ¿Sale todavía el pico en el periodograma?
2. Cambia el ruido a 0,0003 (una estrella más brillante). ¿Qué es lo más pequeño que encuentras?
3. Con la masa de la velocidad radial y el radio del tránsito, calcula la densidad del planeta en g/cm³ y compárala con la de Júpiter (1,33) y la Tierra (5,5).

In [ ]:
# Tu experimento aquí